# Lung X-Ray Classification with VGG16

Binary image classifier for lung X-ray images using VGG16 transfer learning (frozen base + fine-tuning of the last block), with Grad-CAM visual explanations.

- Framework: TensorFlow / Keras
- Data: a public lung X-ray dataset (add the source link here). Expected layout: `train/` (and optionally `val/`, `test/`) with one sub-folder per class, or class folders directly under the root.
- Output: classification report, confusion matrix, saved model, single-image prediction, Grad-CAM.

> For machine-learning experimentation only. This is **not** a medical device and must not be used for diagnosis.

**How to run:** set `dataset_root` in section 3 and run the cells in order (GPU runtime recommended).

## 1. Connect Google Drive

In [ ]:
# Colab only: mount Google Drive (skip if your data is local)
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports

In [ ]:
import os
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import tensorflow as tf

from PIL import Image, UnidentifiedImageError
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

## 3. Dataset Configuration

In [ ]:
dataset_root = '/content/lung_xray_dataset'  # <- change to your dataset folder
sample_image_path = ''
compare_image_path_1 = ''
compare_image_path_2 = ''
model_save_path = 'lung_xray_vgg16_best.keras'
cleaned_dataset_root = '/content/lung_xray_cleaned_dataset'

img_height = 224
img_width = 224
batch_size = 32
epochs = 20
fine_tune_epochs = 5
learning_rate = 1e-4
validation_split = 0.2
seed = 42
allowed_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}

print('Dataset root exists:', os.path.exists(dataset_root))
if os.path.exists(dataset_root):
    print('Top-level contents:', os.listdir(dataset_root))

## 4. Resolve Dataset Layout

The notebook supports common folder layouts. If `train`, `val`, or `test` folders are missing, it will try to use the class folders directly.

In [ ]:
def find_first_existing(root, names):
    for name in names:
        path = os.path.join(root, name)
        if os.path.isdir(path):
            return path
    return None

train_dir = find_first_existing(dataset_root, ['train', 'Train', 'training', 'Training'])
val_dir = find_first_existing(dataset_root, ['val', 'Val', 'valid', 'Valid', 'validation', 'Validation'])
test_dir = find_first_existing(dataset_root, ['test', 'Test', 'testing', 'Testing'])

direct_class_folders = []
if os.path.isdir(dataset_root):
    for item in os.listdir(dataset_root):
        item_path = os.path.join(dataset_root, item)
        if os.path.isdir(item_path) and item.lower() not in {'train', 'training', 'test', 'testing', 'val', 'valid', 'validation'}:
            direct_class_folders.append(item)

if train_dir is None and len(direct_class_folders) >= 2:
    train_dir = dataset_root

print('train_dir:', train_dir)
print('val_dir:', val_dir)
print('test_dir:', test_dir)
print('direct_class_folders:', direct_class_folders)

if train_dir is None:
    raise ValueError('Could not identify a training directory. Check the dataset folder structure.')

## 5. Show Files and Class Distribution

In [ ]:
def get_class_counts(folder_path):
    counts = {}
    if folder_path is None or not os.path.isdir(folder_path):
        return counts
    for class_name in sorted(os.listdir(folder_path)):
        class_path = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_path):
            continue
        image_files = [
            file_name for file_name in os.listdir(class_path)
            if Path(file_name).suffix.lower() in allowed_extensions
        ]
        counts[class_name] = len(image_files)
    return counts

train_counts = get_class_counts(train_dir)
val_counts = get_class_counts(val_dir)
test_counts = get_class_counts(test_dir)

print('Train counts:', train_counts)
print('Validation counts:', val_counts)
print('Test counts:', test_counts)

display(pd.DataFrame({
    'train': pd.Series(train_counts),
    'val': pd.Series(val_counts),
    'test': pd.Series(test_counts)
}).fillna(0).astype(int))

## 6. Data Cleaning Check

This step checks for invalid file extensions and corrupted images. It does not delete anything automatically.

In [ ]:
def scan_dataset_issues(folder_path):
    invalid_extension_files = []
    corrupted_files = []

    if folder_path is None or not os.path.isdir(folder_path):
        return invalid_extension_files, corrupted_files

    for root, _, files in os.walk(folder_path):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            suffix = Path(file_name).suffix.lower()

            if suffix not in allowed_extensions:
                invalid_extension_files.append(file_path)
                continue

            try:
                with Image.open(file_path) as img:
                    img.verify()
            except (UnidentifiedImageError, OSError, SyntaxError):
                corrupted_files.append(file_path)

    return invalid_extension_files, corrupted_files

invalid_train, corrupted_train = scan_dataset_issues(train_dir)
invalid_val, corrupted_val = scan_dataset_issues(val_dir)
invalid_test, corrupted_test = scan_dataset_issues(test_dir)

all_invalid = invalid_train + invalid_val + invalid_test
all_corrupted = corrupted_train + corrupted_val + corrupted_test

print('Invalid extension files:', len(all_invalid))
print('Corrupted image files:', len(all_corrupted))

if all_invalid[:5]:
    print('\nExamples of invalid extension files:')
    for path in all_invalid[:5]:
        print(path)

if all_corrupted[:5]:
    print('\nExamples of corrupted files:')
    for path in all_corrupted[:5]:
        print(path)

## 7. Optional Cleaning Copy

If corrupted files are found, run this cell to copy only valid images into a clean working folder inside Colab.

In [ ]:
create_clean_copy = False

def copy_valid_images_only(source_dir, destination_dir):
    if source_dir is None or not os.path.isdir(source_dir):
        return

    for root, _, files in os.walk(source_dir):
        relative_root = os.path.relpath(root, source_dir)
        target_root = os.path.join(destination_dir, relative_root)
        os.makedirs(target_root, exist_ok=True)

        for file_name in files:
            file_path = os.path.join(root, file_name)
            suffix = Path(file_name).suffix.lower()
            if suffix not in allowed_extensions:
                continue
            try:
                with Image.open(file_path) as img:
                    img.verify()
                shutil.copy2(file_path, os.path.join(target_root, file_name))
            except (UnidentifiedImageError, OSError, SyntaxError):
                pass

if create_clean_copy:
    if os.path.exists(cleaned_dataset_root):
        shutil.rmtree(cleaned_dataset_root)
    os.makedirs(cleaned_dataset_root, exist_ok=True)
    copy_valid_images_only(dataset_root, cleaned_dataset_root)
    dataset_root = cleaned_dataset_root
    print('Using cleaned dataset copy:', dataset_root)
else:
    print('Skipped clean copy. Set create_clean_copy = True only if needed.')

## 8. Show Sample Images

In [ ]:
def collect_sample_images(folder_path, max_per_class=3):
    sample_paths = []
    if folder_path is None or not os.path.isdir(folder_path):
        return sample_paths

    for class_name in sorted(os.listdir(folder_path)):
        class_path = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_path):
            continue
        class_files = [
            os.path.join(class_path, file_name)
            for file_name in sorted(os.listdir(class_path))
            if Path(file_name).suffix.lower() in allowed_extensions
        ]
        for file_path in class_files[:max_per_class]:
            sample_paths.append((class_name, file_path))
    return sample_paths

preview_source = train_dir if train_dir != dataset_root else dataset_root
sample_paths = collect_sample_images(preview_source, max_per_class=3)

plt.figure(figsize=(12, 8))
for i, (class_name, file_path) in enumerate(sample_paths[:9]):
    plt.subplot(3, 3, i + 1)
    img = Image.open(file_path).convert('RGB')
    plt.imshow(img)
    plt.title(class_name)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 9. Create Data Generators

Images are resized to `224 x 224` and normalized with `rescale=1./255`. Training data uses augmentation. Validation and test data do not.

In [ ]:
train_dir = find_first_existing(dataset_root, ['train', 'Train', 'training', 'Training']) or dataset_root
val_dir = find_first_existing(dataset_root, ['val', 'Val', 'valid', 'Valid', 'validation', 'Validation'])
test_dir = find_first_existing(dataset_root, ['test', 'Test', 'testing', 'Testing'])

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=validation_split if val_dir is None and test_dir is None else 0.0
)

eval_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    validation_split=validation_split if val_dir is None and test_dir is None else 0.0
)

In [ ]:
if val_dir is None and test_dir is None:
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        class_mode='binary',
        subset='training',
        shuffle=True,
        seed=seed
    )

    val_generator = eval_datagen.flow_from_directory(
        train_dir,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        class_mode='binary',
        subset='validation',
        shuffle=False,
        seed=seed
    )

    test_generator = None
else:
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        class_mode='binary',
        shuffle=True,
        seed=seed
    )

    eval_source_dir = val_dir if val_dir is not None else test_dir
    val_generator = eval_datagen.flow_from_directory(
        eval_source_dir,
        target_size=(img_height, img_width),
        batch_size=batch_size,
        class_mode='binary',
        shuffle=False,
        seed=seed
    )

    test_generator = None
    if test_dir is not None and val_dir is not None:
        test_generator = eval_datagen.flow_from_directory(
            test_dir,
            target_size=(img_height, img_width),
            batch_size=batch_size,
            class_mode='binary',
            shuffle=False,
            seed=seed
        )

class_indices = train_generator.class_indices
index_to_class = {value: key for key, value in class_indices.items()}

print('Class indices:', class_indices)
print('Training samples:', train_generator.samples)
print('Validation samples:', val_generator.samples)
if test_generator is not None:
    print('Test samples:', test_generator.samples)

## 10. Preview a Processed Batch

In [ ]:
images, labels = next(train_generator)
plt.figure(figsize=(12, 8))
for i in range(min(9, len(images))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    plt.title(index_to_class[int(labels[i])])
    plt.axis('off')
plt.tight_layout()
plt.show()

print('Batch shape:', images.shape)
print('Label shape:', labels.shape)

## 11. Build the Model

VGG16 already contains convolution blocks, `same` padding behavior, and max-pooling layers. We use it as a feature extractor and add a binary classifier head on top.

In [ ]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(img_height, img_width, 3)
)

for layer in base_model.layers:
    layer.trainable = False

model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.summary()

## 12. Show Pooling Layers Inside VGG16

In [ ]:
for layer in base_model.layers:
    if 'pool' in layer.name.lower():
        print(layer.name, layer.output.shape)

## 13. Compile the Model

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=learning_rate),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7),
    ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True)
]

## 14. Train the Model

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs,
    callbacks=callbacks
)

## 15. Fine-Tune the Last VGG16 Layers

In [ ]:
for layer in base_model.layers[:-4]:
    layer.trainable = False
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=fine_tune_epochs,
    callbacks=callbacks
)

## 16. Plot Training Curves

In [ ]:
all_accuracy = history.history['accuracy'] + history_fine.history['accuracy']
all_val_accuracy = history.history['val_accuracy'] + history_fine.history['val_accuracy']
all_loss = history.history['loss'] + history_fine.history['loss']
all_val_loss = history.history['val_loss'] + history_fine.history['val_loss']

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(all_accuracy, label='Train Accuracy')
plt.plot(all_val_accuracy, label='Validation Accuracy')
plt.axvline(x=len(history.history['accuracy']) - 1, color='gray', linestyle='--', label='Fine-tuning start')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(all_loss, label='Train Loss')
plt.plot(all_val_loss, label='Validation Loss')
plt.axvline(x=len(history.history['loss']) - 1, color='gray', linestyle='--', label='Fine-tuning start')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 17. Evaluate the Model

In [ ]:
evaluation_generator = test_generator if test_generator is not None else val_generator
evaluation_name = 'test' if test_generator is not None else 'validation'

eval_loss, eval_accuracy = model.evaluate(evaluation_generator, verbose=1)
print(f'{evaluation_name.title()} Loss: {eval_loss:.4f}')
print(f'{evaluation_name.title()} Accuracy: {eval_accuracy:.4f}')

## 18. Classification Report and Confusion Matrix

In [ ]:
pred_probs = model.predict(evaluation_generator)
pred_labels = (pred_probs > 0.5).astype(int).ravel()
true_labels = evaluation_generator.classes
target_names = [index_to_class[i] for i in sorted(index_to_class.keys())]

print(classification_report(true_labels, pred_labels, target_names=target_names))

cmatrix = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cmatrix, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix ({evaluation_name.title()})')
plt.show()

## 19. Single-Image Prediction

This cell gives a clear prediction label, raw probability, and confidence score for one X-ray image.

In [ ]:
def preprocess_single_image(image_path, target_size=(img_height, img_width)):
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img, img_array

def predict_xray(image_path):
    img, img_array = preprocess_single_image(image_path)
    prob = float(model.predict(img_array, verbose=0)[0][0])

    predicted_index = 1 if prob > 0.5 else 0
    predicted_class = index_to_class[predicted_index]
    confidence = prob if predicted_index == 1 else 1 - prob

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Prediction: {predicted_class}\nConfidence: {confidence:.4f}')
    plt.show()

    print(f'Image: {image_path}')
    print(f'Predicted class: {predicted_class}')
    print(f'Raw probability: {prob:.4f}')
    print(f'Confidence: {confidence:.4f}')

    if predicted_class.lower() in ['cancer', 'malignant', 'lung_opacity', 'opacity']:
        print('Result: Model suspects cancer or abnormality in this X-ray.')
    else:
        print('Result: Model predicts non-cancer or normal for this X-ray.')

    return prob, predicted_class, confidence

if sample_image_path and os.path.exists(sample_image_path):
    predict_xray(sample_image_path)
else:
    print('Set sample_image_path to a real image path before running this cell.')

## 20. Compare Two X-Rays

This cell compares two images and tells you which one looks more suspicious to the model.

In [ ]:
def compare_two_xrays(image_path_1, image_path_2):
    prob1, class1, conf1 = predict_xray(image_path_1)
    prob2, class2, conf2 = predict_xray(image_path_2)

    print('\nComparison Summary')
    print('-' * 40)
    print(f'Image 1: {class1} | confidence={conf1:.4f} | raw_prob={prob1:.4f}')
    print(f'Image 2: {class2} | confidence={conf2:.4f} | raw_prob={prob2:.4f}')

    if prob1 > prob2:
        print('Image 1 looks more suspicious to the model.')
    elif prob2 > prob1:
        print('Image 2 looks more suspicious to the model.')
    else:
        print('Both images look equally suspicious to the model.')

if compare_image_path_1 and compare_image_path_2:
    if os.path.exists(compare_image_path_1) and os.path.exists(compare_image_path_2):
        compare_two_xrays(compare_image_path_1, compare_image_path_2)
    else:
        print('One or both comparison image paths do not exist.')
else:
    print('Set compare_image_path_1 and compare_image_path_2 before running this cell.')

## 21. Grad-CAM Heatmap and Suspicious Side Hint

This section gives a visual explanation. It can hint whether the left or right side looks more suspicious to the model, but it is not a medical diagnosis.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name='block5_conv3'):
    """Grad-CAM for the VGG16 + dense head model built above."""
    vgg_model = model.layers[0]
    conv_model = Model(vgg_model.input, [vgg_model.get_layer(last_conv_layer_name).output, vgg_model.output])
    head_layers = model.layers[1:]

    with tf.GradientTape() as tape:
        conv_outputs, x = conv_model(img_array)
        tape.watch(conv_outputs)
        for layer in head_layers:
            x = layer(x)
        loss = x[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.squeeze(conv_outputs[0] @ pooled_grads[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / tf.math.maximum(tf.reduce_max(heatmap), tf.keras.backend.epsilon())
    return heatmap.numpy()

def show_gradcam_with_side_hint(image_path, last_conv_layer_name='block5_conv3'):
    original_img, img_array = preprocess_single_image(image_path)
    prob = float(model.predict(img_array, verbose=0)[0][0])
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)

    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (img_height, img_width)).numpy().squeeze()

    left_score = float(heatmap_resized[:, :img_width // 2].mean())
    right_score = float(heatmap_resized[:, img_width // 2:].mean())

    # Note: in a standard chest X-ray the patient's left lung appears on the RIGHT side of the image.
    if left_score > right_score:
        side_text = 'The model focuses more on the LEFT half of the image.'
    elif right_score > left_score:
        side_text = 'The model focuses more on the RIGHT half of the image.'
    else:
        side_text = 'The model focuses on both halves equally.'

    img_np = np.array(original_img) / 255.0
    heatmap_color = cm.jet(heatmap_resized)[:, :, :3]
    overlay = np.clip(0.6 * img_np + 0.4 * heatmap_color, 0, 1)

    plt.figure(figsize=(12, 4))
    for i, (im, title, cmap) in enumerate([(original_img, 'Original X-ray', None),
                                          (heatmap_resized, 'Grad-CAM Heatmap', 'jet'),
                                          (overlay, 'Overlay', None)]):
        plt.subplot(1, 3, i + 1)
        plt.imshow(im, cmap=cmap); plt.title(title); plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'Model probability for class "{index_to_class[1]}": {prob:.4f}')
    print(side_text)
    print('Note: this is only a visual explanation of the model, not a medical diagnosis.')

if sample_image_path and os.path.exists(sample_image_path):
    show_gradcam_with_side_hint(sample_image_path)
else:
    print('Set sample_image_path to a real image path before running this cell.')

## 22. Save the Final Model

In [ ]:
model.save(model_save_path)
print(f'Model saved to: {model_save_path}')

## Results

_Fill this in after running the notebook:_ accuracy, precision/recall per class and the confusion matrix.

## Limitations

- Single public dataset; no external validation.
- Grad-CAM shows where the model looks, not medical evidence.
- Research/education only, not for clinical use.